# Span-Free Information Extraction with aibackends + GLiNER2.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/aibackends/blob/main/examples/notebooks/gliner25_extraction_colab.ipynb)

This notebook demos the **extraction backend** built on
[GLiNER2.5](https://fastino.ai/blog/gliner2-5-span-free-information-extraction), a
span-free information-extraction model family from Fastino. Instead of enumerating
every possible span (which capped older GLiNER models at ~12-word entities), GLiNER2.5
scores span *boundaries*, which unlocks:

- entities of **any length** (full clauses, long quotes, complete addresses)
- **long documents** via built-in chunking with offsets remapped to the source text
- **joint entity + relation** extraction as one globally consistent graph
- **constrained classification** with declarative implies/excludes rules
- **span attributes** (negation, sentiment, status) decoded in the same forward pass

Everything runs **entirely on your machine** — no extraction API, no data leaving the
runtime.

| Variant | Checkpoint | Params | Notes |
|---|---|---|---|
| `small` | `fastino/gliner2.5-small-v1` | 74M | fastest, great on CPU |
| `base` | `fastino/gliner2.5-base-v1` | 194M | default, best English quality |
| `multi` | `fastino/gliner2.5-multi-v1` | 287M | multilingual |

**What this notebook covers**

1. Load once, reuse everywhere (cold vs. warm cost)
2. Entity extraction with natural-language labels + offset-based redaction
3. Unlimited span length
4. Span attributes (clinical notes)
5. Constrained classification (agent routing)
6. Joint entity–relation graphs
7. Long documents
8. Native batching + multilingual NER
9. The same tasks from the CLI

> Runs fine on a **free CPU runtime**. A GPU runtime works too and is picked up
> automatically.

## Setup

The `extraction` extra pulls in `gliner2[local]` and `protobuf`.

In [1]:
import importlib.util

if importlib.util.find_spec('aibackends') is None:
    %pip install -q 'aibackends[extraction]'

# If a later import fails with a protobuf or transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import time

import aibackends
from aibackends.backends.extraction import get_extraction_backend, list_extraction_backends

print('aibackends', aibackends.__version__)
print('extraction backends:', list_extraction_backends())

# The backend accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch
    DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

print('device:', DEVICE)

aibackends 0.5.0
extraction backends: ['gliner2.5']


device: cpu


## 1. Load once, reuse everywhere

Models are cached per process, per checkpoint, **and** per device, so you pay the load
cost once. `model` accepts `small` / `base` / `multi` or any Hugging Face repo id.

The first run of the cell below also downloads ~400MB of weights from the Hugging Face
Hub.

In [3]:
backend = get_extraction_backend('gliner2.5')

t = time.perf_counter()
backend.load(model='base', device=DEVICE)
load_s = time.perf_counter() - t
print(f'model load: {load_s:.1f}s')

model load: 4.6s


## 2. Entity extraction with natural-language labels

`extract_entities` takes labels as a list, or as a dict of `label: description` when
the name alone is ambiguous. Every span comes back with half-open character offsets
into the source text — `text[start:end] == entity.text`, guaranteed.

| Field | Meaning |
|---|---|
| `label` | which of your labels matched |
| `text` | the extracted span |
| `start`, `end` | character offsets into the input |
| `confidence` | model confidence for the span |
| `attributes` | per-span attribute values (section 4) |

In [4]:
from aibackends.tasks import extract_entities

message = (
    'Please update the billing contact to Maria Gonzales, reachable at '
    'maria.gonzales@example.com or +1 (404) 555-0182. Ship the replacement '
    'card ending 4111 1111 1111 1111 to 4800 Lakeside Commons Drive, Suite '
    '1200, Atlanta, Georgia 30339.'
)

pii_labels = {
    'person_name': 'Full names of people',
    'email_address': 'Email addresses',
    'phone_number': 'Phone numbers in any format',
    'postal_address': 'Complete postal or street addresses',
    'credit_card_number': 'Payment card numbers',
}

t = time.perf_counter()
pii = extract_entities(message, labels=pii_labels, model='base', device=DEVICE,
                       threshold=0.4)
warm_ms = (time.perf_counter() - t) * 1000

for ent in pii.entities:
    print(f'[{ent.label}] chars {ent.start}..{ent.end} '
          f'(conf {ent.confidence:.2f}): {ent.text}')
print(f'\nwarm call: {warm_ms:.0f}ms  (vs {load_s * 1000:.0f}ms to load)')

[person_name] chars 37..51 (conf 1.00): Maria Gonzales
[email_address] chars 66..92 (conf 1.00): maria.gonzales@example.com
[phone_number] chars 96..113 (conf 1.00): +1 (404) 555-0182
[credit_card_number] chars 148..167 (conf 1.00): 4111 1111 1111 1111
[postal_address] chars 171..234 (conf 0.99): 4800 Lakeside Commons Drive, Suite 1200, Atlanta, Georgia 30339

warm call: 115ms  (vs 4583ms to load)


Exact offsets mean redaction happens at the source — no re-searching for the matched
string, no double-replacement bugs:

In [5]:
redacted = message
for ent in sorted(pii.entities, key=lambda e: e.start or 0, reverse=True):
    redacted = redacted[:ent.start] + f'[{ent.label.upper()}]' + redacted[ent.end:]
print(redacted)

Please update the billing contact to [PERSON_NAME], reachable at [EMAIL_ADDRESS] or [PHONE_NUMBER]. Ship the replacement card ending [CREDIT_CARD_NUMBER] to [POSTAL_ADDRESS].


## 3. Unlimited span length

GLiNER2 enumerated candidate spans up to ~12 words; longer entities were structurally
invisible. The boundary architecture removes the cap — a 25-word quote costs the same
to locate as a 2-word name.

In [6]:
earnings = (
    'During the earnings call, CEO Amara Osei said "we expect revenue to '
    'grow by twenty percent next year driven by strong demand in our cloud '
    'division and continued expansion into Southeast Asian markets", before '
    'asking investors to send written questions to Meridian Cloud Holdings, '
    'Investor Relations, 4800 Lakeside Commons Drive, Suite 1200, Atlanta, '
    'Georgia 30339, United States.'
)

long_spans = extract_entities(
    earnings,
    labels={
        'quote': 'The complete quoted statement',
        'postal_address': 'A complete postal address including street, city, and country',
        'person_name': 'Full names of people',
    },
    model='base', device=DEVICE, threshold=0.4,
)

for ent in long_spans.entities:
    words = len(ent.text.split())
    cap = "  <- beyond GLiNER2's ~12-word cap" if words > 12 else ''
    print(f'[{ent.label}] {words} words (conf {ent.confidence:.2f}){cap}')
    print(f'  {ent.text}')
    assert earnings[ent.start:ent.end] == ent.text

[person_name] 2 words (conf 1.00)
  Amara Osei
[quote] 25 words (conf 1.00)  <- beyond GLiNER2's ~12-word cap
  we expect revenue to grow by twenty percent next year driven by strong demand in our cloud division and continued expansion into Southeast Asian markets
[postal_address] 11 words (conf 0.99)
  4800 Lakeside Commons Drive, Suite 1200, Atlanta, Georgia 30339, United States


## 4. Span attributes

Attribute groups qualify each extracted span **in the same forward pass** — no second
classification model over the extracted spans. Here symptoms carry negation status and
medications carry an active/discontinued status.

In [7]:
note = (
    'Patient denies chest pain but reports severe headache and intermittent '
    'dizziness. Prescribed 400mg ibuprofen twice daily for the headache. '
    'Aspirin was discontinued last month due to a mild allergy.'
)

clinical = extract_entities(
    note,
    labels={
        'symptom': 'Symptoms or complaints mentioned for the patient',
        'medication': 'Names of drugs or pharmaceutical substances',
        'dosage': 'Dose amounts such as 400mg or 2 tablets',
    },
    attributes={
        'negation': {'labels': ['present', 'denied by patient'],
                     'applies_to': ['symptom']},
        'status': {'labels': ['currently prescribed', 'discontinued'],
                   'applies_to': ['medication']},
    },
    model='base', device=DEVICE, threshold=0.4,
)

for ent in clinical.entities:
    quals = ', '.join(f'{g}={a.label}' for g, a in ent.attributes.items())
    print(f'[{ent.label}] {ent.text}' + (f'  ({quals})' if quals else ''))

[symptom] chest pain  (negation=denied by patient)
[symptom] severe headache  (negation=present)
[symptom] intermittent dizziness  (negation=present)
[dosage] 400mg
[medication] ibuprofen  (status=currently prescribed)
[symptom] headache  (negation=present)
[medication] Aspirin  (status=discontinued)


## 5. Constrained classification

`classify_text` decodes one or more classification tasks jointly, under declarative
rules (`implies`, `excludes`, `iff`). Contradictory outputs become **impossible by
construction** — no downstream reconciliation code. `feasible` tells you whether the
constraints could be satisfied at all.

In [8]:
from aibackends.tasks import classify_text

routing_tasks = {
    'intent': {'labels': ['read', 'write', 'delete']},
    'effects': {'labels': ['read_only', 'create', 'modify', 'delete'],
                'multi_label': True, 'min_labels': 1, 'max_labels': 2},
}
rules = [
    {'kind': 'implies', 'when': ['intent', 'delete'], 'then': ['effects', 'delete']},
    {'kind': 'implies', 'when': ['intent', 'read'], 'then': ['effects', 'read_only']},
    {'kind': 'excludes', 'when': ['intent', 'read'], 'then': ['effects', 'delete']},
    {'kind': 'excludes', 'when': ['intent', 'read'], 'then': ['effects', 'modify']},
]

requests = [
    'Delete the temporary files from /tmp before the backup runs',
    'Preview the quarterly report without changing anything',
    'Append the new customer records to the ledger',
]

for request in requests:
    verdict = classify_text(request, tasks=routing_tasks, constraints=rules,
                            model='base', device=DEVICE)
    print(f'{request[:52]:54} intent={verdict.value("intent"):6} '
          f'effects={verdict.values("effects")} feasible={verdict.feasible}')

Delete the temporary files from /tmp before the back   intent=delete effects=['delete'] feasible=True


Preview the quarterly report without changing anythi   intent=read   effects=['read_only'] feasible=True
Append the new customer records to the ledger          intent=write  effects=['create'] feasible=True


## 6. Joint entity–relation extraction

`extract_graph` decodes entities and typed relations **together** as one consistent
graph: every relation endpoint exists, endpoint types are enforced (`works_for` goes
person → organization), and schema rules like `unique_head` (one employer per person)
hold by construction.

In [9]:
from aibackends.tasks import extract_graph

graph = extract_graph(
    'Tim Cook leads Apple in Cupertino. Sundar Pichai runs Google in Mountain View.',
    entities=['person', 'organization', 'location'],
    relations=[
        {'name': 'works_for', 'head': 'person', 'tail': 'organization',
         'unique_head': True},
        {'name': 'located_in', 'head': 'organization', 'tail': 'location'},
    ],
    model='base', device=DEVICE,
)

print(f'feasible: {graph.feasible}\n')
for ent in graph.entities:
    print(f'  {ent.id}: [{ent.type}] {ent.text}')
print()
for head, rel, tail in graph.triples():
    print(f'  {head} -{rel}-> {tail}')

feasible: True

  e1: [person] Tim Cook
  e2: [organization] Apple
  e3: [location] Cupertino
  e4: [person] Sundar Pichai
  e5: [organization] Google
  e6: [location] Mountain View

  Apple -located_in-> Cupertino
  Google -located_in-> Mountain View
  Tim Cook -works_for-> Apple
  Sundar Pichai -works_for-> Google


## 7. Long documents

`long_document=True` splits the text into overlapping word chunks, extracts per chunk,
remaps every span back to character offsets in the **original** document, and merges
duplicates across overlaps. Below, a full master services agreement.

In [10]:
from pathlib import Path
from urllib.request import urlopen

local = Path('../data/sample_contract.txt')
if local.exists():
    contract = local.read_text(encoding='utf-8')
else:
    url = ('https://raw.githubusercontent.com/donvito/aibackends/main/'
           'examples/data/sample_contract.txt')
    contract = urlopen(url).read().decode('utf-8')

print(f'{len(contract)} characters, {len(contract.split())} words')

deal = extract_entities(
    contract,
    labels={
        'party': 'Named companies or organizations that are parties to the agreement',
        'monetary_amount': 'Money amounts such as USD 84,500',
        'duration': 'Time periods such as twenty-four months or ninety days',
    },
    model='base', device=DEVICE, threshold=0.5,
    long_document=True, chunk_size=384, chunk_overlap=64,
)

for ent in deal.entities:
    assert contract[ent.start:ent.end] == ent.text

unique = sorted({(e.label, e.text) for e in deal.entities})
for label, text in unique:
    print(f'  [{label}] {text}')
print(f'\n{len(deal.entities)} spans extracted, all offsets verified against the source.')

5863 characters, 857 words


  [duration] forty-five days
  [duration] ninety days
  [duration] seventy-two
hours
  [duration] sixty days
  [duration] thirty days
  [duration] twelve-month periods
  [duration] twenty-four months
  [monetary_amount] USD 84,500
  [party] Beacon Retail Group
  [party] Beacon Retail Group, Inc.
  [party] Customer
  [party] Northwind
Analytics GmbH
  [party] Northwind Analytics GmbH
  [party] Provider

49 spans extracted, all offsets verified against the source.


## 8. Native batching + multilingual NER

`extract_entities_batch` runs one model batch instead of a Python loop. Pair it with
the `multi` checkpoint and English labels work across languages.

In [11]:
import pandas as pd

from aibackends.tasks import extract_entities_batch

texts = [
    'La empresa Iberdrola anunció una inversión de 3.000 millones de euros '
    'en Valencia junto a su presidente Ignacio Galán.',
    'Die Lufthansa eröffnet ein neues Drehkreuz in München, wie '
    'Vorstandschef Carsten Spohr am Montag erklärte.',
    "L'entreprise TotalEnergies a signé un accord avec le gouvernement du "
    'Sénégal à Dakar, selon Patrick Pouyanné.',
]

t = time.perf_counter()
batch = extract_entities_batch(
    texts,
    labels=['person', 'organization', 'location', 'monetary amount'],
    model='multi', device=DEVICE, threshold=0.4, batch_size=4,
)
batch_s = time.perf_counter() - t

rows = [
    {
        'text': text[:48] + '...',
        'entities': '; '.join(f'{e.label}: {e.text}' for e in result.entities),
    }
    for text, result in zip(texts, batch, strict=True)
]
print(f'{len(texts)} texts in {batch_s:.2f}s (first multi-model call includes load)\n')
pd.DataFrame(rows)

3 texts in 4.94s (first multi-model call includes load)



,text,entities
0,La empresa Iberdrola anunció una inversión de ...,organization: Iberdrola; monetary amount: 3.00...
1,Die Lufthansa eröffnet ein neues Drehkreuz in ...,organization: Lufthansa; location: München; pe...
2,L'entreprise TotalEnergies a signé un accord a...,organization: TotalEnergies; organization: gou...


## 9. From the CLI

Same tasks, no Python. `aibackends task` prints JSON, so it pipes into `jq` nicely.

In [12]:
!aibackends task extract-entities --input 'Apple hired Jane Doe in London.' --labels company,person,location --model small

{
  "text": "Apple hired Jane Doe in London.",
  "entities": [
    {
      "label": "company",
      "text": "Apple",
      "start": 0,
      "end": 5,
      "confidence": 0.9985211491584778,
      "attributes": {}
    },
    {
      "label": "person",
      "text": "Jane Doe",
      "start": 12,
      "end": 20,
      "confidence": 0.9959174990653992,
      "attributes": {}
    },
    {
      "label": "location",
      "text": "London",
      "start": 24,
      "end": 30,
      "confidence": 0.9993370175361633,
      "attributes": {}
    }
  ],
  "backend_used": "gliner2.5",
  "model_id": "fastino/gliner2.5-small-v1"
}


In [13]:
!aibackends task extract-graph --input 'Alice works for Acme in Paris.' --entities person,organization,location --relation works_for:person:organization --relation located_in:organization:location --model small

{
  "text": "Alice works for Acme in Paris.",
  "entities": [
    {
      "id": "e1",
      "type": "person",
      "text": "Alice",
      "start": 0,
      "end": 5,
      "confidence": 0.9998359825976328
    },
    {
      "id": "e2",
      "type": "organization",
      "text": "Acme",
      "start": 16,
      "end": 20,
      "confidence": 0.9997808354265485
    },
    {
      "id": "e3",
      "type": "location",
      "text": "Paris",
      "start": 24,
      "end": 29,
      "confidence": 0.9999983629853888
    }
  ],
  "relations": [
    {
      "type": "located_in",
      "head": "e2",
      "tail": "e3",
      "head_text": "Acme",
      "tail_text": "Paris",
      "confidence": 0.7671723233410082
    },
    {
      "type": "works_for",
      "head": "e1",
      "tail": "e2",
      "head_text": "Alice",
      "tail_text": "Acme",
      "confidence": 0.8642189109101701
    }
  ],
  "feasible": true,
  "backend_used": "gliner2.5",
  "model_id": "fastino/gliner2.5-small-v1"
}


## Recap

```python
from aibackends.tasks import extract_entities, classify_text, extract_graph

extract_entities(text, labels={...}, attributes={...}, long_document=True)
classify_text(text, tasks={...}, constraints=[...])
extract_graph(text, entities=[...], relations=[...])
```

- Three checkpoints: `small` (74M) / `base` (194M) / `multi` (287M, multilingual)
- Spans of any length, verified character offsets, attributes in one pass
- Graphs and classifications that are consistent by construction (`feasible` flag)
- Runnable scripts for every capability live in `examples/gliner25/`
- CPU latency and zero-shot accuracy reports live in `benchmarks/reports/` and
  `evals/reports/`